# Exploratory Data Analysis: Smoke Detection Dataset

**Dataset**: Sensor Fusion Smoke Detection Classification  
**Source**: [Kaggle](https://www.kaggle.com/datasets/gauravduttakiit/sensorfusion-smoke-detection-classification)  
**Date**: 2025-12-16

---

## Objectives

1. **Understand sensor characteristics**: What are the distributions, ranges, and correlations?
2. **Temporal patterns**: How do sensors evolve over time?
3. **Fire vs. Normal**: What distinguishes alarm events?
4. **Missing data**: Are there gaps or sensor failures?
5. **Visualization challenges**: How can we display 14 sensors effectively?

---

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Plotting configuration
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.precision', 3)

## 1. Load Data

**Note**: Download dataset manually from Kaggle and place in `../datasets/` directory.

In [ ]:
# Load data (update path as needed)
data_path = '../datasets/smoke_detection_iot.csv'

try:
    df = pd.read_csv(data_path)
    print(f"✓ Loaded {len(df):,} samples")
except FileNotFoundError:
    print(f"✗ Dataset not found at {data_path}")
    print("Please download from: https://www.kaggle.com/datasets/gauravduttakiit/sensorfusion-smoke-detection-classification")
    df = None

## 2. Initial Exploration

In [ ]:
if df is not None:
    # Basic info
    print("Dataset Shape:", df.shape)
    print("\nColumn Names:")
    print(df.columns.tolist())
    print("\nFirst 5 rows:")
    display(df.head())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isnull().sum())

## 3. Sensor Statistics

In [ ]:
if df is not None:
    # Identify sensor columns (exclude UTC, CNT, Fire Alarm)
    sensor_cols = [col for col in df.columns if col not in ['UTC', 'CNT', 'Fire Alarm']]
    
    print(f"Found {len(sensor_cols)} sensor channels:")
    print(sensor_cols)
    
    # Summary statistics
    print("\nSummary Statistics:")
    display(df[sensor_cols].describe())

## 4. Class Distribution

In [ ]:
if df is not None and 'Fire Alarm' in df.columns:
    # Fire alarm distribution
    alarm_counts = df['Fire Alarm'].value_counts()
    
    print("Fire Alarm Distribution:")
    print(alarm_counts)
    print(f"\nPercentage of fire events: {alarm_counts[1] / len(df) * 100:.2f}%")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(8, 5))
    alarm_counts.plot(kind='bar', ax=ax, color=['green', 'red'])
    ax.set_xlabel('Fire Alarm')
    ax.set_ylabel('Count')
    ax.set_title('Class Distribution: Fire vs. Normal')
    ax.set_xticklabels(['Normal (0)', 'Fire (1)'], rotation=0)
    plt.tight_layout()
    plt.show()

## 5. Time-Series Visualization

**Challenge**: Display 14 sensors evolving over time without overwhelming the viewer.

In [ ]:
if df is not None:
    # Sample subset for visualization (first 1000 samples)
    df_sample = df.head(1000).copy()
    
    # Convert UTC to datetime if available
    if 'UTC' in df_sample.columns:
        df_sample['timestamp'] = pd.to_datetime(df_sample['UTC'], unit='s')
    else:
        df_sample['timestamp'] = df_sample['CNT']  # Use counter as proxy
    
    # Group sensors by type
    temp_humidity = ['Temperature[C]', 'Humidity[%]'] if 'Temperature[C]' in df_sample.columns else []
    gas_sensors = ['TVOC[ppb]', 'eCO2[ppm]', 'Raw H2', 'Raw Ethanol'] if 'TVOC[ppb]' in df_sample.columns else []
    particle_sensors = ['PM1.0', 'PM2.5', 'NC0.5', 'NC1.0', 'NC2.5'] if 'PM1.0' in df_sample.columns else []
    
    # Plot temperature & humidity
    if temp_humidity:
        fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
        
        for i, col in enumerate(temp_humidity):
            if col in df_sample.columns:
                axes[i].plot(df_sample['timestamp'], df_sample[col], label=col)
                axes[i].set_ylabel(col)
                axes[i].legend()
                axes[i].grid(True, alpha=0.3)
        
        axes[-1].set_xlabel('Time')
        fig.suptitle('Temperature & Humidity Over Time', fontsize=14)
        plt.tight_layout()
        plt.show()

## 6. Correlation Analysis

**Question**: Which sensors are most correlated? Could we reduce dimensionality?

In [ ]:
if df is not None:
    # Correlation matrix
    corr_matrix = df[sensor_cols].corr()
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 10))
    sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
                center=0, square=True, ax=ax, cbar_kws={'label': 'Correlation'})
    ax.set_title('Sensor Correlation Matrix', fontsize=14)
    plt.tight_layout()
    plt.show()
    
    # Find highly correlated pairs
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.8:
                high_corr.append((corr_matrix.columns[i], corr_matrix.columns[j], corr_matrix.iloc[i, j]))
    
    if high_corr:
        print("\nHighly correlated sensor pairs (|r| > 0.8):")
        for s1, s2, r in high_corr:
            print(f"  {s1} <-> {s2}: {r:.3f}")

## 7. Fire Event Analysis

**Question**: What do sensors look like just before a fire alarm?

In [ ]:
if df is not None and 'Fire Alarm' in df.columns:
    # Compare fire vs. normal distributions
    fire_df = df[df['Fire Alarm'] == 1]
    normal_df = df[df['Fire Alarm'] == 0]
    
    print(f"Fire samples: {len(fire_df)}")
    print(f"Normal samples: {len(normal_df)}")
    
    # Plot distributions for key sensors
    key_sensors = ['Temperature[C]', 'PM2.5', 'TVOC[ppb]'] if all(s in df.columns for s in ['Temperature[C]', 'PM2.5', 'TVOC[ppb]']) else sensor_cols[:3]
    
    fig, axes = plt.subplots(1, len(key_sensors), figsize=(15, 4))
    
    for i, sensor in enumerate(key_sensors):
        if sensor in df.columns:
            axes[i].hist(normal_df[sensor], bins=50, alpha=0.6, label='Normal', color='green', density=True)
            axes[i].hist(fire_df[sensor], bins=50, alpha=0.6, label='Fire', color='red', density=True)
            axes[i].set_xlabel(sensor)
            axes[i].set_ylabel('Density')
            axes[i].legend()
            axes[i].grid(True, alpha=0.3)
    
    fig.suptitle('Sensor Distributions: Fire vs. Normal', fontsize=14)
    plt.tight_layout()
    plt.show()

## 8. Next Steps

Based on this EDA, we can:

1. **Feature engineering**: Create derived features (e.g., rate of change, rolling averages)
2. **Sensor selection**: Remove highly correlated sensors to reduce dimensionality
3. **Kalman filtering**: Smooth noisy sensors (Temperature, Humidity)
4. **Anomaly detection**: Identify outliers or sensor failures
5. **Classification models**: Predict fire alarm from sensor readings
6. **Temporal fusion**: How do sensors evolve in the 30s before alarm?

---

**Continue to**: `02_sensor_fusion_kalman.ipynb`